# HRP Database Setup

This notebook does the following:
1. Loading health monitoring data from Excel files
2. Setting up a local SQLite database for the data storage
3. Transforming and normalizing 17-sheets measurement data

**Summary of Results:**
- **Main Data File**: 17 parameter sheets (~1M rows each)
- **Medical Info**: 7,130 seniors with disease & medication data
- **SOS Alerts**: 1,092 alert records
- **Storage**: Local SQLite database

In [ ]:
import sys
import os
import time
from pathlib import Path
import warnings

import pandas as pd

sys.path.append(os.path.abspath(".."))
from src.utils.database import initialize_database

warnings.filterwarnings('ignore')

## Section 1: Load and Inspect Raw Excel Data

Load the three Excel files and examine their structure, data types, and content.

In [12]:
# Define paths
raw_data_dir = Path("../data/raw/HRP")
data_file = raw_data_dir / "data_202511181045.xlsx"
med_file = raw_data_dir / "Med&Dis_202511181011.xlsx"
sos_file = raw_data_dir / "SOS_202511181012.xlsx"

assert data_file.exists()
assert med_file.exists()
assert sos_file.exists()

In [13]:
# Load Medications & Disease Data
df_medical = pd.read_excel(med_file, engine="openpyxl")
df_medical.shape

(7130, 3)

In [6]:
df_medical.columns

Index(['seniorID', 'diseaseNames', 'medicineNames'], dtype='object')

In [7]:
df_medical.dtypes

seniorID          int64
diseaseNames     object
medicineNames    object
dtype: object

In [8]:
df_medical.head()

,seniorID,diseaseNames,medicineNames
0,2875,"Osteoporoza,Nadciśnienie tętnicze,Arytmia serc...","Acard,Emanera,Agen,Concor,Valzek"
1,3755,"Miażdzyca,Osteoporoza","Gensulin,Beto,Furosemidum,Amlopin,Zahron,Berod..."
2,3762,"Cukrzyca,Niedoczynnośc tarczycy,Niedoczynnośc ...","Letrox,Diosminex,Valsacor,Metformax,Bibloc,Pol..."
3,3805,"Stomia,Niedosłuch,Skolioza","Pregabalin,Staveran,Neurovit"
4,4367,"Miażdżyca kończyn dolnych,Niewydolnośc układu ...","Allupol,Cipropol,Eliquis,Ezehron,Areplex"


In [9]:
df_medical.isnull().sum()

seniorID         0
diseaseNames     0
medicineNames    0
dtype: int64

In [14]:
# Load SOS Alerts
df_sos = pd.read_excel(sos_file, engine="openpyxl")
df_sos.shape

(1092, 3)

In [11]:
df_sos.columns

Index(['seniorID', 'alertDate', 'sosNote'], dtype='object')

In [12]:
df_sos.dtypes

seniorID              int64
alertDate    datetime64[ns]
sosNote              object
dtype: object

In [13]:
df_sos.head()

,seniorID,alertDate,sosNote
0,3275,2025-11-14 15:01:34,Alarm przypadkowy
1,3792,2025-11-12 11:17:06,Alarm przypadkowy
2,3794,2025-11-13 11:31:14,Alarm przypadkowy
3,3794,2025-11-13 11:10:03,Alarm przypadkowy
4,4365,2025-11-12 22:18:55,Alert techniczny


In [14]:
df_sos.isnull().sum()

seniorID     0
alertDate    0
sosNote      2
dtype: int64

In [15]:
# Measurement data sheets
xls = pd.ExcelFile(data_file)
sheet_names = xls.sheet_names
print(f"Total sheets: {len(sheet_names)}")
print(f"Sheet names: {sheet_names}\n")

Total sheets: 18
Sheet names: ['expdata', 'expdata#1', 'expdata#2', 'expdata#3', 'expdata#4', 'expdata#5', 'expdata#6', 'expdata#7', 'expdata#8', 'expdata#9', 'expdata#10', 'expdata#11', 'expdata#12', 'expdata#13', 'expdata#14', 'expdata#15', 'expdata#16', 'expdata#17']



In [35]:
for i, sheet_name in enumerate(sheet_names[7:9]):
    df = pd.read_excel(data_file, sheet_name=sheet_name, nrows=5, engine="openpyxl")
    print(f"\nSheet '{sheet_name}':")
    print(f"    Col 0 (seniorID): {df.iloc[:, 0].values[:2]}")
    print(f"    Col 1 (value): {df.iloc[:, 1].values[:2]}")
    print(f"    Col 2 (sbp): {df.iloc[:, 2].values[:2]}")
    print(f"    Col 3 (dbp): {df.iloc[:, 3].values[:2]}")
    print(f"    Col 4 (date): {df.iloc[:, 4].values[:2]}")
    print(f"    Col 5 (type): {df.iloc[:, 5].values[:2]}")


Sheet 'expdata#7':
    Col 0 (seniorID): [47883 44473]
    Col 1 (value): [60 99]
    Col 2 (sbp): [nan nan]
    Col 3 (dbp): [nan nan]
    Col 4 (date): ['2025-11-12T10:00:11.000000000' '2025-11-12T10:00:11.000000000']
    Col 5 (type): ['Heartrate' 'Heartrate']

Sheet 'expdata#8':
    Col 0 (seniorID): [50444 35244]
    Col 1 (value): [nan nan]
    Col 2 (sbp): [131 147]
    Col 3 (dbp): [81 79]
    Col 4 (date): ['2025-11-13T19:07:11.000000000' '2025-11-13T19:07:11.000000000']
    Col 5 (type): ['BloodPressure' 'BloodPressure']


## Section 2: Initialize SQLite Database

Create a local SQLite database with a schema for measurements, medical info, and alerts.

In [ ]:
db_path = Path("../db/hrp_data.db")
db_path.parent.mkdir(parents=True, exist_ok=True)

# Close any prior connection to release file handle (Windows locks the file)
if "conn" in locals() and conn:
    try:
        conn.close()
        print("Closed prior database connection")
    except Exception as e:
        print(f"Warning while closing prior connection: {e}")

# Delete existing database for fresh start
# if db_path.exists():
#     db_path.unlink()
#     print(f"Deleted existing database")

Closed prior database connection
Deleted existing database


In [24]:
# Create new database with schema
conn = initialize_database(db_path)
print(f"Database initialized at {db_path.absolute()}")
print(f"Database size: {db_path.stat().st_size / 1024:.1f} KB")

INFO:src.utils.database:Database initialized at ..\db\hrp_data.db


Database initialized at c:\Users\eldar\Projects\AI-CVD\notebooks\..\db\hrp_data.db
Database size: 3542500.0 KB


In [7]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"\nTables created: {[t[0] for t in tables]}")


Tables created: ['seniors', 'measurements', 'sqlite_sequence', 'medical_info', 'diseases', 'medicines', 'senior_diseases', 'senior_medicines', 'alerts']


## Section 3: Store Data in SQLite Database

Load data from Excel files and insert into SQLite with proper normalization and indexing.

In [18]:
# Rename columns to match database schema
df_medical_renamed = df_medical.rename(columns={
    'seniorID': 'senior_id',
    'diseaseNames': 'disease_names',
    'medicineNames': 'medicine_names'
})

df_sos_renamed = df_sos.rename(columns={
    'seniorID': 'senior_id',
    'alertDate': 'alert_date',
    'sosNote': 'sos_note'
})

In [10]:
# Medical information
print(f"Total records: {len(df_medical_renamed)}")
print(f"Unique senior_ids: {df_medical_renamed['senior_id'].nunique()}")

Total records: 7130
Unique senior_ids: 7129


In [20]:
# Deduplicate & keep the last entry for each senior_id
if df_medical_renamed['senior_id'].duplicated().any():
    duplicate_count = df_medical_renamed['senior_id'].duplicated().sum()
    print(f"Found {duplicate_count} duplicate senior_ids - keeping last entry for each")
    
    duplicated_ids = df_medical_renamed[df_medical_renamed['senior_id'].duplicated(keep=False)].sort_values('senior_id')
    print(f"\nDuplicate entries (showing all {len(duplicated_ids)} rows):")
    print(duplicated_ids.to_string())
    
    df_medical_renamed = df_medical_renamed.drop_duplicates(subset=['senior_id'], keep='last')
    print(f"\nAfter deduplication: {len(df_medical_renamed)} records")

df_medical_renamed.to_sql("medical_info", conn, if_exists="replace", index=False)
print(f"Inserted {len(df_medical_renamed)} medical records")

Found 1 duplicate senior_ids - keeping last entry for each

Duplicate entries (showing all 2 rows):
      senior_id                                                                                                                                                                                                                       disease_names                                                                                  medicine_names
4842      46877  Nadciśnienie tętnicze,Arytmia serca,Zaćma,Dyskopatia,Zwyrodnienie kręgosłupa,Żylaki,Niedowidzenie,Jaskra,Miażdzyca,Osteoporoza,Niewydolnośc układu krążenia,Hipercholesterolemia,Migotanie przedsionków,Choroba wieńcowa,Reumatyzm  Betaserc,Coronal,Suvardio,Preductal,Cardilopin,Prestarium,Memotropil,Preductal,Cavinton,Tobrex
4843      46877  Nadciśnienie tętnicze,Arytmia serca,Zaćma,Dyskopatia,Zwyrodnienie kręgosłupa,Żylaki,Niedowidzenie,Jaskra,Miażdzyca,Osteoporoza,Niewydolnośc układu krążenia,Hipercholesterolemia,Migotanie przedsionków,Cho

In [21]:
print(f"Total alert records: {len(df_sos_renamed)}")
df_sos_renamed.to_sql("alerts", conn, if_exists="replace", index=False)
print(f"Inserted {len(df_sos_renamed)} alert records")

Total alert records: 1092
Inserted 1092 alert records


In [22]:
# Load and normalize measurement data
all_measurements = []
total_rows = 0

for i, sheet_name in enumerate(sheet_names, 1):
    print(f"  [{i:2d}/{len(sheet_names)}] Loading '{sheet_name}'...", end=" ", flush=True)
    
    # Load sheet
    df = pd.read_excel(data_file, sheet_name=sheet_name, engine="openpyxl")
    initial_rows = len(df)
    
    # Extract columns according to structure: seniorID, value, sbp, dbp, date, type
    df_normalized = pd.DataFrame({
        "senior_id": df.iloc[:, 0],
        "value": df.iloc[:, 1],
        "sbp": df.iloc[:, 2],
        "dbp": df.iloc[:, 3],
        "date": df.iloc[:, 4],
        "type": df.iloc[:, 5]
    })
    
    # Convert data types
    df_normalized["senior_id"] = pd.to_numeric(df_normalized["senior_id"], errors="coerce")
    df_normalized["value"] = pd.to_numeric(df_normalized["value"], errors="coerce")
    df_normalized["sbp"] = pd.to_numeric(df_normalized["sbp"], errors="coerce")
    df_normalized["dbp"] = pd.to_numeric(df_normalized["dbp"], errors="coerce")
    df_normalized["type"] = df_normalized["type"].astype(str).str.strip()
    
    # Remove rows with NULL senior_id
    df_normalized = df_normalized.dropna(subset=["senior_id"])
    
    all_measurements.append(df_normalized)
    total_rows += len(df_normalized)
    
    print(f"{len(df_normalized):,} rows")

print(f"\nTotal normalized measurement rows: {total_rows:,}")

  [ 1/18] Loading 'expdata'... 1,048,575 rows
  [ 2/18] Loading 'expdata#1'... 1,048,575 rows
  [ 3/18] Loading 'expdata#2'... 1,048,575 rows
  [ 4/18] Loading 'expdata#3'... 1,048,575 rows
  [ 5/18] Loading 'expdata#4'... 1,048,575 rows
  [ 6/18] Loading 'expdata#5'... 1,048,575 rows
  [ 7/18] Loading 'expdata#6'... 1,048,575 rows
  [ 8/18] Loading 'expdata#7'... 1,048,575 rows
  [ 9/18] Loading 'expdata#8'... 

KeyboardInterrupt: 

In [90]:
# Combine all measurement data
df_measurements = pd.concat(all_measurements, ignore_index=True)
df_measurements.shape

(1048575, 6)

In [91]:
df_measurements.head(10)

,senior_id,value,sbp,dbp,date,type
0,36280,36.6,NaN,NaN,2025-11-13 09:17:37,Temperature
1,26847,36.6,NaN,NaN,2025-11-13 09:18:03,Temperature
2,47228,36.6,NaN,NaN,2025-11-13 09:18:10,Temperature
3,41278,36.6,NaN,NaN,2025-11-13 09:18:11,Temperature
4,18331,36.6,NaN,NaN,2025-11-13 09:18:11,Temperature
5,46340,36.6,NaN,NaN,2025-11-13 09:18:11,Temperature
6,12766,36.6,NaN,NaN,2025-11-13 09:18:11,Temperature
7,33548,36.4,NaN,NaN,2025-11-13 09:18:11,Temperature
8,9225,36.6,NaN,NaN,2025-11-13 09:18:11,Temperature
9,38226,36.6,NaN,NaN,2025-11-13 09:18:11,Temperature


In [92]:
df_measurements.dtypes

senior_id             int64
value               float64
sbp                 float64
dbp                 float64
date         datetime64[ns]
type                 object
dtype: object

In [93]:
df_measurements['type'].unique()

array(['Temperature'], dtype=object)

In [ ]:
print("Inserting measurements in batches...")
batch_size = 50000
for i in range(0, len(df_measurements), batch_size):
    batch = df_measurements.iloc[i:i+batch_size]
    batch.to_sql("measurements", conn, if_exists="append", index=False)
    if (i // batch_size + 1) % 10 == 0:
        print(f"  Inserted {min(i + batch_size, len(df_measurements)):,} measurements...")

conn.commit()
print(f" All {len(df_measurements):,} measurements inserted successfully")

Inserting measurements in batches...
  ✓ Inserted 500,000 measurements...
  ✓ Inserted 500,000 measurements...
  ✓ Inserted 1,000,000 measurements...
  ✓ Inserted 1,000,000 measurements...
  ✓ Inserted 1,500,000 measurements...
  ✓ Inserted 2,000,000 measurements...
  ✓ Inserted 2,500,000 measurements...
  ✓ Inserted 3,000,000 measurements...
  ✓ Inserted 3,500,000 measurements...
  ✓ Inserted 4,000,000 measurements...
  ✓ Inserted 4,500,000 measurements...
  ✓ Inserted 5,000,000 measurements...
  ✓ Inserted 5,500,000 measurements...
  ✓ Inserted 6,000,000 measurements...
  ✓ Inserted 6,500,000 measurements...
  ✓ Inserted 7,000,000 measurements...
  ✓ Inserted 7,500,000 measurements...
  ✓ Inserted 8,000,000 measurements...
  ✓ Inserted 8,500,000 measurements...
  ✓ Inserted 9,000,000 measurements...
  ✓ Inserted 9,500,000 measurements...
  ✓ Inserted 10,000,000 measurements...
  ✓ Inserted 10,500,000 measurements...
  ✓ Inserted 11,000,000 measurements...
  ✓ Inserted 11,500,000 meas

In [26]:
# Display database file size
db_size_mb = db_path.stat().st_size / (1024**2)
print(f"\nDB Size: {db_size_mb:.2f} MB")


DB Size: 3459.47 MB


In [ ]:
# Show Normalized tables
cursor = conn.cursor()
counts = {
    "seniors": cursor.execute("SELECT COUNT(*) FROM seniors").fetchone()[0],
    "measurements": cursor.execute("SELECT COUNT(*) FROM measurements").fetchone()[0],
    "alerts": cursor.execute("SELECT COUNT(*) FROM alerts").fetchone()[0],
    "medical_info": cursor.execute("SELECT COUNT(*) FROM medical_info").fetchone()[0],
    "diseases": cursor.execute("SELECT COUNT(*) FROM diseases").fetchone()[0],
    "medicines": cursor.execute("SELECT COUNT(*) FROM medicines").fetchone()[0],
    "senior_diseases": cursor.execute("SELECT COUNT(*) FROM senior_diseases").fetchone()[0],
    "senior_medicines": cursor.execute("SELECT COUNT(*) FROM senior_medicines").fetchone()[0],
}

print("\nRow counts after normalization:")
for table, count in counts.items():
    print(f"  {table:.<20} {count:>15,}")


Row counts after normalization:
  seniors.............          11,908
  measurements........      18,033,186
  alerts..............           1,092
  medical_info........           7,129
  diseases............             161
  medicines...........           1,626
  senior_diseases.....          41,879
  senior_medicines....          41,699


## Section 4: Verify Data Integrity

Check that all data was correctly transferred to the database and no data loss occurred.

In [57]:
# Verify row counts
cursor = conn.cursor()

counts = {
    "measurements": cursor.execute("SELECT COUNT(*) FROM measurements").fetchone()[0],
    "medical_info": cursor.execute("SELECT COUNT(*) FROM medical_info").fetchone()[0],
    "alerts": cursor.execute("SELECT COUNT(*) FROM alerts").fetchone()[0],
}

print("\n Row Count Verification:")
for table, count in counts.items():
    print(f"  {table:.<30} {count:>15,}")


 Row Count Verification:
  measurements..................      18,033,186
  medical_info..................           7,129
  alerts........................           1,092


In [58]:
# Verify unique seniors
print("\n Unique Seniors:")
unique_seniors = cursor.execute("SELECT COUNT(DISTINCT senior_id) FROM measurements").fetchone()[0]
print(f"  Total unique seniors: {unique_seniors:,}")


 Unique Seniors:
  Total unique seniors: 11,908
  Total unique seniors: 11,908


In [27]:
# Verify no NULL values in columns
print("\n NULL Value Checks:")
null_checks = {
    "measurements.senior_id": cursor.execute("SELECT COUNT(*) FROM measurements WHERE senior_id IS NULL").fetchone()[0],
    "measurements.date": cursor.execute("SELECT COUNT(*) FROM measurements WHERE date IS NULL").fetchone()[0],
    "measurements.type": cursor.execute("SELECT COUNT(*) FROM measurements WHERE type IS NULL").fetchone()[0],
}

for check, null_count in null_checks.items():
    status = "OK -" if null_count == 0 else "WARNING"
    print(f"  {status} {check:.<35} {null_count} NULLs")


 NULL Value Checks:
  OK - measurements.senior_id............. 0 NULLs
  OK - measurements.date.................. 0 NULLs
  OK - measurements.type.................. 0 NULLs


In [28]:
# Check data types and ranges
cursor.execute("""
    SELECT 
        type,
        COUNT(*) as count,
        COUNT(DISTINCT senior_id) as unique_seniors,
        MIN(value) as min_val,
        MAX(value) as max_val,
        AVG(value) as avg_val
    FROM measurements
    WHERE value IS NOT NULL
    GROUP BY type
    ORDER BY count DESC
""")

In [29]:
print("Data Distribution by Measurement Type")
for mtype, count, uniq_seniors, min_val, max_val, avg_val in cursor.fetchall():
    print(f"  {mtype:.<20} {count:>10,} rows | {uniq_seniors:>6,} seniors | {min_val:>8.2f} to {max_val:>8.2f} (avg: {avg_val:.2f})")

Data Distribution by Measurement Type
  Heartrate...........  4,074,735 rows | 11,762 seniors |    24.00 to   205.00 (avg: 73.42)
  Temperature.........  4,057,151 rows | 11,744 seniors |    36.30 to   127.90 (avg: 36.72)
  Saturation..........  3,250,993 rows | 11,708 seniors |    80.00 to    99.00 (avg: 97.08)
  Steps...............  2,575,576 rows | 10,168 seniors |     1.00 to 36955.00 (avg: 3173.54)


In [30]:
# Check blood pressure data
print("\nNo. of Rows with BP data:")
bp_count = cursor.execute("SELECT COUNT(*) FROM measurements WHERE sbp IS NOT NULL OR dbp IS NOT NULL").fetchone()[0]
bp_count


No. of Rows with BP data:


4074731

In [88]:
# Verify timestamp formats
print("\nDate/Timestamp Verification:")
cursor.execute("SELECT date FROM measurements LIMIT 5")
sample_timestamps = cursor.fetchall()
for ts in sample_timestamps:
    print(f"  Sample: {ts[0]}")


Date/Timestamp Verification:
  Sample: 2025-11-10 00:00:00
  Sample: 2025-11-10 00:00:01
  Sample: 2025-11-10 00:00:09
  Sample: 2025-11-10 00:00:10
  Sample: 2025-11-10 00:00:10


In [67]:
# Check date range
cursor.execute("SELECT MIN(date), MAX(date) FROM measurements")
min_date, max_date = cursor.fetchone()
print(f"\n  Date range: {min_date} to {max_date}")


  Date range: 2025-11-10 00:00:00 to 2025-11-15 23:59:56


In [31]:
# Check for duplicate measurements
duplicates = cursor.execute("""
    SELECT senior_id, date, type, COUNT(*) as dup_count
    FROM measurements
    GROUP BY senior_id, date, type
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchall()

In [32]:
print("\nDuplicate Check:")
if duplicates:
    print(f"  Found {len(duplicates)} potential duplicates (first 5):")
    for sid, date, mtype, count in duplicates:
        print(f"    senior_id={sid}, date={date}, type={mtype} appears {count} times")
else:
    print("  No duplicates were found")


Duplicate Check:
  Found 5 potential duplicates (first 5):
    senior_id=4442, date=2025-11-14 06:48:30, type=BloodPressure appears 2 times
    senior_id=4442, date=2025-11-14 06:48:30, type=Heartrate appears 2 times
    senior_id=8708, date=2025-11-13 06:07:11, type=BloodPressure appears 2 times
    senior_id=8708, date=2025-11-13 06:07:11, type=Heartrate appears 2 times
    senior_id=8709, date=2025-11-10 10:57:16, type=BloodPressure appears 2 times


## Section 5: Query and Validate Stored Data

Execute SQL queries to retrieve data and perform basic analysis to confirm database functionality.

### Example 1: Get all measurements for a specific type

In [78]:
# Example 1: Get Heartrate measurements (first 10) 
query1 = """
    SELECT senior_id, value, date, type
    FROM measurements
    WHERE type = 'Heartrate'
    ORDER BY date
    LIMIT 10
"""

In [79]:
df_example1 = pd.read_sql(query1, conn)
df_example1

,senior_id,value,date,type
0,11605,57.0,2025-11-10 00:00:10,Heartrate
1,49771,60.0,2025-11-10 00:00:10,Heartrate
2,29761,47.0,2025-11-10 00:00:10,Heartrate
3,30706,65.0,2025-11-10 00:00:10,Heartrate
4,50444,49.0,2025-11-10 00:00:10,Heartrate
5,43643,90.0,2025-11-10 00:00:10,Heartrate
6,38602,92.0,2025-11-10 00:00:10,Heartrate
7,22340,60.0,2025-11-10 00:00:10,Heartrate
8,23961,91.0,2025-11-10 00:00:10,Heartrate
9,6727,66.0,2025-11-10 00:00:10,Heartrate


### Example 2: Aggregate statistics by measurement type

In [80]:
# Example 2: Aggregate statistics by measurement type
query2 = """
    SELECT 
        type,
        COUNT(*) as measurement_count,
        COUNT(DISTINCT senior_id) as unique_seniors,
        AVG(value) as avg_value,
        MIN(value) as min_value,
        MAX(value) as max_value,
        ROUND(AVG(value), 2) as mean
    FROM measurements
    WHERE value IS NOT NULL
    GROUP BY type
    ORDER BY measurement_count DESC
"""

In [81]:
df_example2 = pd.read_sql(query2, conn)
df_example2

,type,measurement_count,unique_seniors,avg_value,min_value,max_value,mean
0,Heartrate,4074735,11762,73.424053,24.0,205.0,73.42
1,Temperature,4057151,11744,36.718686,36.3,127.9,36.72
2,Saturation,3250993,11708,97.076295,80.0,99.0,97.08
3,Steps,2575576,10168,3173.538838,1.0,36955.0,3173.54


### Example 3: Get measurements for a specific senior

In [94]:
# Find a sample senior ID first
sample_senior_id = int(df_measurements.iloc[0]["senior_id"])
query3 = """
    SELECT senior_id, value, sbp, dbp, date, type
    FROM measurements
    WHERE senior_id = ?
    ORDER BY date DESC
    LIMIT 10
"""

In [95]:
df_example3 = pd.read_sql(query3, conn, params=[sample_senior_id])
df_example3

,senior_id,value,sbp,dbp,date,type
0,36280,99.0,NaN,NaN,2025-11-15 23:56:11,Saturation
1,36280,NaN,138.0,89.0,2025-11-15 23:56:11,BloodPressure
2,36280,64.0,NaN,NaN,2025-11-15 23:56:11,Heartrate
3,36280,36.6,NaN,NaN,2025-11-15 23:56:11,Temperature
4,36280,99.0,NaN,NaN,2025-11-15 23:46:11,Saturation
5,36280,NaN,134.0,89.0,2025-11-15 23:46:11,BloodPressure
6,36280,64.0,NaN,NaN,2025-11-15 23:46:11,Heartrate
7,36280,36.8,NaN,NaN,2025-11-15 23:46:11,Temperature
8,36280,99.0,NaN,NaN,2025-11-15 23:36:11,Saturation
9,36280,NaN,142.0,92.0,2025-11-15 23:36:11,BloodPressure


### Example 4: Query performance test

In [98]:
# Get all Heartrate measurements
start = time.time()
query4 = "SELECT * FROM measurements WHERE type = 'Heartrate' LIMIT 1000"
df_example4 = pd.read_sql(query4, conn)
elapsed = time.time() - start

In [99]:
print(f"  Retrieved {len(df_example4)} rows in {elapsed:.4f} seconds")

  Retrieved 1000 rows in 0.0254 seconds


In [100]:
df_example4.head(10)

,id,senior_id,value,sbp,dbp,date,type
0,6562468,11605,57.0,None,None,2025-11-10 00:00:10,Heartrate
1,6566465,49771,60.0,None,None,2025-11-10 00:00:10,Heartrate
2,6581644,29761,47.0,None,None,2025-11-10 00:00:10,Heartrate
3,6582055,30706,65.0,None,None,2025-11-10 00:00:10,Heartrate
4,6582056,50444,49.0,None,None,2025-11-10 00:00:10,Heartrate
5,6586749,43643,90.0,None,None,2025-11-10 00:00:10,Heartrate
6,6615195,38602,92.0,None,None,2025-11-10 00:00:10,Heartrate
7,6622152,22340,60.0,None,None,2025-11-10 00:00:10,Heartrate
8,6622153,23961,91.0,None,None,2025-11-10 00:00:10,Heartrate
9,6626011,6727,66.0,None,None,2025-11-10 00:00:10,Heartrate


### Example 4: Blood Pressure Analysis

In [102]:
query5 = """
    SELECT senior_id, sbp, dbp, date, type
    FROM measurements
    WHERE sbp IS NOT NULL AND dbp IS NOT NULL
    ORDER BY date DESC
    LIMIT 10
"""

In [103]:
df_example5 = pd.read_sql(query5, conn)
df_example5

,senior_id,sbp,dbp,date,type
0,9263,128.0,55.0,2025-11-15 23:59:53,BloodPressure
1,50494,112.0,89.0,2025-11-15 23:59:38,BloodPressure
2,26794,120.0,78.0,2025-11-15 23:59:33,BloodPressure
3,4043,151.0,57.0,2025-11-15 23:59:32,BloodPressure
4,48782,133.0,83.0,2025-11-15 23:59:32,BloodPressure
5,30983,126.0,72.0,2025-11-15 23:59:32,BloodPressure
6,49493,124.0,84.0,2025-11-15 23:59:31,BloodPressure
7,47346,119.0,75.0,2025-11-15 23:59:31,BloodPressure
8,21960,112.0,70.0,2025-11-15 23:59:31,BloodPressure
9,40467,132.0,75.0,2025-11-15 23:59:31,BloodPressure


### Example 5: Get Blood Pressure Statistics

In [104]:
query5_stats = """
    SELECT 
        COUNT(*) as bp_measurements,
        COUNT(DISTINCT senior_id) as seniors_with_bp,
        AVG(sbp) as avg_systolic,
        AVG(dbp) as avg_diastolic,
        MIN(sbp) as min_systolic,
        MAX(sbp) as max_systolic,
        MIN(dbp) as min_diastolic,
        MAX(dbp) as max_diastolic
    FROM measurements
    WHERE sbp IS NOT NULL AND dbp IS NOT NULL
"""

In [105]:
df_bp_stats = pd.read_sql(query5_stats, conn)
df_bp_stats

,bp_measurements,seniors_with_bp,avg_systolic,avg_diastolic,min_systolic,max_systolic,min_diastolic,max_diastolic
0,4074731,11762,129.392577,78.623184,68.0,212.0,23.0,156.0
